In [1]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.express as px


import matplotlib.pyplot as plt

In [2]:
history = pd.read_csv("lead_90th_history_cleaned.csv")

replacement = pd.read_csv("clean_LSLR.csv",)

inventory = pd.read_csv("clean_inventory.csv")

In [3]:
print(history.columns.tolist())
print(replacement.columns.tolist())
print(inventory.columns.tolist())

['pwsid', 'system_name', 'county', 'population', 'monitoring_end_date', 'lead_90th_ppb', 'includes_5th_liter_or_not', 'sampling_next_due_subject_to_change', 'year', 'month']
['pwsid', 'year', 'county_for_checking', 'lines_replaced']
['pwsid', 'supply_name', 'lead_lines', 'gpcl_lines', 'non_lead_lines', 'unknown_lines', 'total_lines', 'calc_total', 'total_mismatch', 'inventory_status']


In [4]:
inventory["year"] = 2025

In [5]:
history_subset = history[
    [
        "pwsid",
        "year",
        "system_name",
        "county",
        "population",
        "lead_90th_ppb",
        "monitoring_end_date",
        "month"
    ]
]

In [6]:
trend_df = history.merge(
    replacement,
    on=["pwsid", "year"],
    how="left"
)

In [7]:
print("history rows:", len(history))
print("trend_df rows:", len(trend_df))

history rows: 6616
trend_df rows: 6616


In [8]:
print(history.duplicated(subset=["pwsid", "year"]).sum())

1137


In [9]:
print(replacement.duplicated(subset=["pwsid", "year"]).sum())

0


In [10]:
print(inventory.duplicated(subset=["pwsid", "year"]).sum())

0


In [11]:
print(trend_df["lines_replaced"].notna().sum())
print(trend_df["lines_replaced"].isna().sum())

778
5838


In [12]:
check_inv_rep = inventory.merge(
    replacement,
    on=["pwsid", "year"],
    how="left",
    indicator=True
)

print(check_inv_rep["_merge"].value_counts())

_merge
left_only     1383
right_only       0
both             0
Name: count, dtype: int64


In [13]:
check_inv_hist = inventory.merge(
    history,
    on=["pwsid", "year"],
    how="left",
    indicator=True
)

print(check_inv_hist["_merge"].value_counts())

_merge
left_only     1383
right_only       0
both             0
Name: count, dtype: int64


In [14]:
print(inventory["pwsid"].dtype)
print(replacement["pwsid"].dtype)
print(history["pwsid"].dtype)

print(inventory["pwsid"].head(10).tolist())
print(replacement["pwsid"].head(10).tolist())
print(history["pwsid"].head(10).tolist())

object
object
object
['MI0000011   ', 'MI0000012   ', 'MI0000020   ', 'MI0000030   ', 'MI0000040   ', 'MI0000045   ', 'MI0000048   ', 'MI0000050   ', 'MI0000070   ', 'MI0000072   ']
['MI0000040', 'MI0000040', 'MI0000040', 'MI0000040', 'MI0000100', 'MI0000100', 'MI0000110', 'MI0000110', 'MI0000120', 'MI0000120']
['MI0040520', 'MI0040520', 'MI0000011', 'MI0000011', 'MI0000011', 'MI0000012', 'MI0000012', 'MI0000012', 'MI0000012', 'MI0000012']


In [15]:
inventory["pwsid"] = inventory["pwsid"].str.strip()
replacement["pwsid"] = replacement["pwsid"].str.strip()
history["pwsid"] = history["pwsid"].str.strip()

In [16]:
check_inv_rep = inventory.merge(
    replacement,
    on=["pwsid", "year"],
    how="left",
    indicator=True
)

print(check_inv_rep["_merge"].value_counts())

_merge
left_only     1383
right_only       0
both             0
Name: count, dtype: int64


In [17]:
check_inv_hist = inventory.merge(
    history,
    on=["pwsid", "year"],
    how="left",
    indicator=True
)

print(check_inv_hist["_merge"].value_counts())

_merge
left_only     1374
both            10
right_only       0
Name: count, dtype: int64


In [18]:
print(sorted(trend_df["year"].unique()))

[np.float64(2015.0), np.float64(2016.0), np.float64(2017.0), np.float64(2018.0), np.float64(2019.0), np.float64(2020.0), np.float64(2021.0), np.float64(2022.0), np.float64(2023.0), np.float64(2024.0), np.float64(2025.0), np.float64(2026.0)]


In [19]:
trend_df["year"] = trend_df["year"].astype(int)

In [20]:
print(sorted(replacement["year"].unique()))

[np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


In [21]:
trend_df["lines_replaced"].isna().sum()

np.int64(5838)

In [22]:
len(trend_df)

6616

In [23]:
trend_df.groupby("year")["lines_replaced"].apply(lambda x: x.isna().sum())

year
2015      4
2016    316
2017    499
2018    504
2019    770
2020    798
2021    605
2022    815
2023    904
2024    612
2025     10
2026      1
Name: lines_replaced, dtype: int64

In [24]:
trend_df.groupby("year").size()

year
2015       4
2016     316
2017     499
2018     504
2019     770
2020     798
2021     795
2022     985
2023    1119
2024     815
2025      10
2026       1
dtype: int64

In [25]:
analysis_df = trend_df[
    (trend_df["year"] >= 2017) &
    (trend_df["year"] <= 2024)
]

In [26]:
lead_trend = (
    analysis_df
    .groupby("year")["lead_90th_ppb"]
    .mean()
    .reset_index()
)

print(lead_trend)

   year  lead_90th_ppb
0  2017       1.933868
1  2018       3.057540
2  2019       5.116883
3  2020       3.676692
4  2021       3.006289
5  2022       3.386802
6  2023       2.647900
7  2024       2.943558


In [27]:
print(inventory.columns)

Index(['pwsid', 'supply_name', 'lead_lines', 'gpcl_lines', 'non_lead_lines',
       'unknown_lines', 'total_lines', 'calc_total', 'total_mismatch',
       'inventory_status', 'year'],
      dtype='object')


In [ ]:
# county list
county_list = sorted(analysis_df["county"].dropna().unique())

county_dropdown = widgets.Dropdown(
    options=county_list,
    description="County:",
    layout=widgets.Layout(width="300px")
)

def update_plot(county):

    df = analysis_df[analysis_df["county"] == county]

    county_trend = (
        df.groupby("year")["lead_90th_ppb"]
        .mean()
        .reset_index()
    )

    fig = px.line(
        county_trend,
        x="year",
        y="lead_90th_ppb",
        markers=True,
        title=f"Lead Level Trend in {county}",
        labels={
            "lead_90th_ppb": "Lead Level (ppb)",
            "year": "Year"
        }
    )

    fig.add_hline(
        y=12,
        line_dash="dash",
        line_color="red",
        annotation_text="EPA Action Level (12 ppb)"
        )

    fig.show()

widgets.interact(update_plot, county=county_dropdown)

interactive(children=(Dropdown(description='County:', layout=Layout(width='300px'), options=('ALCONA', 'ALGER'…

<function __main__.update_plot(county)>

In [29]:
system_options_df = analysis_df[["system_name", "pwsid"]].drop_duplicates().copy()

system_options_df["label"] = (
    system_options_df["system_name"] + " (" + system_options_df["pwsid"] + ")"
)

label_to_pwsid = dict(
    zip(system_options_df["label"], system_options_df["pwsid"])
)

system_dropdown = widgets.Dropdown(
    options=sorted(system_options_df["label"].tolist()),
    description="System:",
    layout=widgets.Layout(width="450px")
)

def update_system_plot(selected_label):
    selected_pwsid = label_to_pwsid[selected_label]

    df = analysis_df[analysis_df["pwsid"] == selected_pwsid].copy()

    df_plot = (
        df.groupby("year", as_index=False)
        .agg(
            lead_90th_ppb=("lead_90th_ppb", "mean"),
            system_name=("system_name", "first"),
            county=("county", "first")
        )
    )

    fig = px.line(
        df_plot,
        x="year",
        y="lead_90th_ppb",
        markers=True,
        title=f"Lead Level Trend: {selected_label}",
        labels={
            "lead_90th_ppb": "Lead Level (ppb)",
            "year": "Year"
        },
        hover_data={
            "system_name": True,
            "county": True,
            "year": True,
            "lead_90th_ppb": ':.2f'
        }
    )

    fig.add_hline(
        y=12,
        line_dash="dash",
        line_color="red",
        annotation_text="Action Level (12 ppb)",
        annotation_position="top left"
    )

    fig.update_layout(
        xaxis_title="Year",
        yaxis_title="Lead Level (ppb)",
        hovermode="x unified"
    )

    fig.show()

widgets.interact(update_system_plot, selected_label=system_dropdown)

interactive(children=(Dropdown(description='System:', layout=Layout(width='450px'), options=('553 MOBILE ESTAT…

<function __main__.update_system_plot(selected_label)>

In [ ]:
system_options_df = analysis_df[["system_name", "pwsid"]].drop_duplicates().copy()

system_options_df["label"] = (
    system_options_df["system_name"] + " (" + system_options_df["pwsid"] + ")"
)

label_to_pwsid = dict(
    zip(system_options_df["label"], system_options_df["pwsid"])
)

replacement_dropdown = widgets.Dropdown(
    options=sorted(system_options_df["label"].tolist()),
    description="System:",
    layout=widgets.Layout(width="450px")
)

def update_replacement_plot(selected_label):
    selected_pwsid = label_to_pwsid[selected_label]

    df = analysis_df[analysis_df["pwsid"] == selected_pwsid].copy()

    df_rep = (
        df[["year", "lines_replaced"]]
        .dropna(subset=["lines_replaced"])
        .groupby("year", as_index=False)["lines_replaced"]
        .sum()
    )

    fig = px.bar(
        df_rep,
        x="year",
        y="lines_replaced",
        title=f"Lead Service Lines Replaced per Year: {selected_label}",
        labels={
            "lines_replaced": "Lines Replaced",
            "year": "Year"
        },
        hover_data={
            "year": True,
            "lines_replaced": ':.0f'
        }
    )

    fig.update_layout(
        xaxis_title="Year",
        yaxis_title="Lines Replaced"
    )

    fig.show()

widgets.interact(update_replacement_plot, selected_label=replacement_dropdown)

interactive(children=(Dropdown(description='System:', layout=Layout(width='450px'), options=('553 MOBILE ESTAT…

<function __main__.update_replacement_plot(selected_label)>

In [ ]:
system_options_df = analysis_df[["system_name", "pwsid"]].drop_duplicates().copy()

system_options_df["label"] = (
    system_options_df["system_name"] + " (" + system_options_df["pwsid"] + ")"
)

label_to_pwsid = dict(
    zip(system_options_df["label"], system_options_df["pwsid"])
)

progress_dropdown = widgets.Dropdown(
    options=sorted(system_options_df["label"].tolist()),
    description="System:",
    layout=widgets.Layout(width="450px")
)

output = widgets.Output()

def update_progress(change):
    selected_label = progress_dropdown.value
    selected_pwsid = label_to_pwsid[selected_label]

    rep_total = analysis_df.loc[
        analysis_df["pwsid"] == selected_pwsid,
        "lines_replaced"
    ].sum()

    inv_match = inventory.loc[inventory["pwsid"] == selected_pwsid]

    with output:
        clear_output()

        print("System:", selected_label)

        if len(inv_match) == 0:
            print("No inventory record found.")
            return

        lead_lines = inv_match["lead_lines"].iloc[0]
        gpcl_lines = inv_match["gpcl_lines"].iloc[0]
        unknown_lines = inv_match["unknown_lines"].iloc[0]

        total_to_identify_or_replace = lead_lines + gpcl_lines + unknown_lines
        denominator = total_to_identify_or_replace + rep_total

        print("Lead lines:", lead_lines)
        print("GPCL lines:", gpcl_lines)
        print("Unknown lines:", unknown_lines)
        print("Total to identify and/or replace:", total_to_identify_or_replace)
        print("Reported replaced lines:", rep_total)

        if denominator > 0:
            percent_replaced = rep_total / denominator
            progress_percent = round(percent_replaced * 100, 1)
            print("Percent replaced:", str(progress_percent) + "%")
        else:
            print("Percent replaced: N/A")

progress_dropdown.observe(update_progress, names="value")

update_progress(None)

display(progress_dropdown)
display(output)


output_chart = widgets.Output()

def update_progress_chart(change):
    selected_label = progress_dropdown.value
    selected_pwsid = label_to_pwsid[selected_label]

    rep_total = analysis_df.loc[
        analysis_df["pwsid"] == selected_pwsid,
        "lines_replaced"
    ].sum()

    inv_match = inventory.loc[inventory["pwsid"] == selected_pwsid]

    with output_chart:
        clear_output()

        if len(inv_match) == 0:
            print("No inventory record found.")
            return

        lead_lines = inv_match["lead_lines"].iloc[0]
        gpcl_lines = inv_match["gpcl_lines"].iloc[0]
        unknown_lines = inv_match["unknown_lines"].iloc[0]

        total_to_identify_or_replace = lead_lines + gpcl_lines + unknown_lines
        denominator = total_to_identify_or_replace + rep_total

        if denominator <= 0:
            print("Not enough data to calculate progress.")
            return

        percent_replaced = rep_total / denominator
        progress_percent = percent_replaced * 100

        fig, ax = plt.subplots(figsize=(8, 1.5))

        ax.barh(
            ["Progress"],
            [denominator],
            alpha=0.3
        )

        ax.barh(
            ["Progress"],
            [rep_total]
        )

        ax.set_xlim(0, denominator)
        ax.set_xlabel("Lines")
        ax.set_title("Percent Replaced")

        ax.text(
            rep_total,
            0,
            "  " + str(round(progress_percent, 1)) + "%",
            va="center"
        )

        plt.show()

progress_dropdown.observe(update_progress_chart, names="value")
update_progress_chart(None)

display(output_chart)

Dropdown(description='System:', layout=Layout(width='450px'), options=('553 MOBILE ESTATES (MI0040520)', 'ACME…

Output()

Output()